# Course 27 lab — Independent quality engineering for AI-2605

Northstar is changing classification and review routing across three repositories. The implementation agent may write code and candidate tests, but it cannot redefine the five protected invariants, edit the trusted oracles, certify its own evidence, or authorize delivery.

## 1. Objectives, constraints, and reproducibility

You will inspect exact Git revisions, execute cross-service behavior, test a confidence-domain property and metamorphic relation, exercise SQLite concurrency, kill mutations, inject failures, compare AI models on development and holdout populations, and evaluate signed evidence. The default path is credential-free and deterministic. Git is required; pytest, Hypothesis, and jsonschema are optional extensions.

The fixture's HMAC keys, checked-in holdout, and local SQLite database are teaching substitutes—not production trust, secrecy, or database-parity claims.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

from quality_app.simulation import quality_context, run_evaluation, run_golden, run_scenario_in_context
from quality_app.test_selector import select_tests

manager = quality_context(ROOT)
context = manager.__enter__()
subject = context['subject']
observations = context['observations']
assert len(subject.component_commits) == 3
dict(subject.component_commits)


The harness copied the three service sources, created reproducible commits, and imported code from those exact files. Evidence also binds contract, oracle, holdout, policy, and suite digests. A passing test against another revision set is not evidence for this subject.

## 2. Baseline: execute actual protected behavior

The baseline is not a collection of `*_enabled` flags. It calls the committed service functions and inspects durable state.

In [ ]:
behavior = observations['behavior']
assert all(behavior['checks'].values())
assert behavior['concurrent_statuses'] == ['APPLIED', 'REPLAYED']
assert behavior['effect_count'] == 1
behavior


Two threads used the same logical operation identity. SQLite's unique key and transaction boundary produced one durable effect; conflicting reuse returned `IDEMPOTENCY_CONFLICT`. This is meaningful for the fixture, but production must repeat the obligation against its real database and queue semantics.

## 3. Property and metamorphic obligations

The protected property is: ambiguity requires review for every confidence value. The metamorphic relation is: increasing confidence cannot remove review while ambiguity is unchanged.

In [ ]:
property_result = observations['property']
metamorphic_result = observations['metamorphic']
assert property_result == {
    'population': 101,
    'failures': [],
    'passed': True,
    'claim': 'enumerated confidence boundary; Hypothesis can expand the generated domain',
}
assert metamorphic_result['passed']
property_result, metamorphic_result


Install the optional tools and run `tests/test_property_hypothesis.py` to let Hypothesis generate and shrink floating-point examples. Generation improves search; the Underwriting Risk-owned property remains the oracle.

## 4. Requirement-weighted mutation testing

The governed denominator contains seven consequential faults. Trivial extra mutations cannot dilute a surviving tenant, authority, idempotency, approval, provenance, ambiguity, or outage defect.

In [ ]:
mutations = observations['mutations']
assert len(mutations) == 7
assert all(item['outcome'] == 'KILLED' for item in mutations)
[(item['mutation_id'], item['requirement_id'], item['detected_by']) for item in mutations]


## 5. Failure injection and named safe states

An exception is not the oracle. The lab asserts zero durable effects after rollback, one effect after duplicate delivery, explicit rejection of stale approval, safe hold during review outage, revalidation after contract change, and quality failure for unsupported evidence.

In [ ]:
faults = observations['fault_experiments']
assert len(faults) == 6
assert all(item['outcome'] == 'SAFE' for item in faults)
[(item['experiment_id'], item['safe_state']) for item in faults]


## 6. AI evaluation: aggregate improvement versus protected failure

The deterministic prediction artifacts separate classification metrics from review-routing and evidence-support invariants. Compare v1 and proposed v2 on the protected holdout.

In [ ]:
comparison = context['evaluation']
v1 = comparison['holdout']['v1']
v2 = comparison['holdout']['v2']
assert v1['accuracy'] == {'numerator': 6, 'denominator': 8, 'value': 0.75}
assert v2['accuracy'] == {'numerator': 8, 'denominator': 8, 'value': 1.0}
assert v1['decision'] == 'CONFORMANT'
assert v2['decision'] == 'BLOCKED'
{'v1': v1['accuracy'], 'v2': v2['accuracy'], 'v2_violations': v2['protected_violations']}


The higher accuracy cannot offset `HOLD-005:REVIEW_ROUTING_BYPASS` or `HOLD-008:UNSUPPORTED_EVIDENCE`. Inspect `per_class`, `confusion_matrix`, and `slices` before making a model decision. These eight holdout records are a teaching population, not a benchmark.

## 7. Evaluation contamination experiment

The `overfit` candidate hardcodes visible development behavior. It appears perfect until the independent population changes.

In [ ]:
overfit_dev = comparison['development']['overfit']['accuracy']
overfit_holdout = comparison['holdout']['overfit']['accuracy']
assert overfit_dev['value'] == 1.0
assert overfit_holdout == {'numerator': 2, 'denominator': 8, 'value': 0.25}
{'development': overfit_dev, 'holdout': overfit_holdout}


## 8. Risk-adaptive selection

Selection adds impact-derived tests but never removes mandatory protected checks. Unknown dependencies require impact review.

In [ ]:
selection = select_tests({
    'category': 'shared_api',
    'protected_boundaries': ['human_review', 'tenant_isolation'],
    'unknown_dependencies': True,
})
assert selection['status'] == 'IMPACT_REVIEW_REQUIRED'
assert {'contract', 'integration', 'property', 'metamorphic', 'security', 'negative'}.issubset(selection['selected_kinds'])
selection


## 9. Independent evidence and decision boundary

The clean receipt covers 17 requirement/kind pairs, seven mutations, and six faults. The gate verifies its producer, signature, exact subject, suite, policy, time, traceability, oracle sources, flake state, and required populations.

In [ ]:
report = context['plane'].evaluate(subject, [context['evidence']], '2026-11-08T18:00:00Z')
assert report.decision == 'CONFORMANT'
assert report.delivery_state == 'READY_FOR_INTEGRATION_REVIEW'
assert not any((report.merge_authorized, report.deployment_authorized, report.activation_authorized, report.release_authorized))
{'decision': report.decision, 'coverage': report.coverage, 'metrics': report.metrics, 'delivery_state': report.delivery_state}


## 10. Failure diagnosis and evaluator performance

Inject a post-execution revision change and then run the fixed scenario populations.

In [ ]:
stale = run_scenario_in_context(context, 'stale-subject')
assert stale['decision'] == 'STALE_EVIDENCE'
golden = run_golden(ROOT, context)
adversarial = run_evaluation(ROOT, context)
assert len(golden) == 15 and all(item['passed'] for item in golden)
assert adversarial['metrics']['exact_decision_accuracy'] == {'numerator': 42, 'denominator': 42, 'value': 1.0}
{'stale_findings': stale['finding_codes'][:3], 'golden': [15, 15], 'adversarial': adversarial['metrics']}


The 15/15 and 42/42 values measure only the registered fixture populations. They do not estimate arbitrary defect detection or production model quality. Clean-control false positives, protected-invariant detection, and exact decision accuracy remain separate metrics.

## 11. Production upgrade and exercises

Replace local HMACs with workload identity, temporary commits with verified source/artifact provenance, checked-in holdout data with access-controlled evaluation services, SQLite with real persistence/queue tests, and local JSON with immutable evidence storage. Add trusted time, revocation, quarantine ownership and expiry, budgets, observability, rollback, and Course 24 release integration.

Exercises: add one invariant end-to-end; create a model with higher average but worse rare-slice recall; introduce an unknown-outcome timeout; design a real-database concurrency test; and write an explicit list of claims this lab cannot prove.

In [ ]:
manager.__exit__(None, None, None)
'temporary Git repositories and SQLite databases cleaned up'
